# XAI Heatmap from Saved Model (No Retraining)

This notebook loads your **saved Keras `.h5` model** and generates an **explainability heatmap** using **Occlusion Sensitivity**.

Why occlusion?
- It only needs `model.predict`.
- It works even when Grad-CAM breaks due to nested model graphs / Keras compatibility.

**Output saved to Drive:** a heatmap + overlay PNG.

In [ ]:
from google.colab import drive
drive.mount('/content/drive')

In [ ]:
import os
import json
import numpy as np
import tensorflow as tf
import matplotlib.pyplot as plt
import cv2

print('TensorFlow:', tf.__version__)
gpus = tf.config.list_physical_devices('GPU')
print('GPU:', gpus[0].name if gpus else 'None')

## 1) Paths (edit if needed)

In [ ]:
# Saved model + class names from your training
MODEL_SAVE_DIR = "/content/drive/MyDrive/models"
MODEL_PATH = f"{MODEL_SAVE_DIR}/MobileNetV2.h5"   # change if using a different model
CLASS_JSON = f"{MODEL_SAVE_DIR}/class_names.json"

# Dataset (optional, used to pick a sample image automatically)
DATASET_DIR = "/content/drive/MyDrive/Plant_leave_diseases_dataset_with_augmentation"

# Output
OUT_DIR = f"{MODEL_SAVE_DIR}/xai_outputs"
os.makedirs(OUT_DIR, exist_ok=True)

IMG_SIZE = (224, 224)

print('MODEL_PATH:', MODEL_PATH)
print('CLASS_JSON:', CLASS_JSON)
print('DATASET_DIR:', DATASET_DIR)
print('OUT_DIR:', OUT_DIR)

## 2) Load class names + model (no retraining)

In [ ]:
from tensorflow.keras import layers, models
from tensorflow.keras.applications import MobileNetV2

with open(CLASS_JSON, 'r', encoding='utf-8') as f:
    obj = json.load(f)
class_names = obj['class_names'] if isinstance(obj, dict) else obj
num_classes = len(class_names)
print('Num classes:', num_classes)

# Loading some .h5 models can fail in newer Colab/Keras (e.g. TrueDivide/InputLayer config).
# Most robust approach: rebuild the architecture and load weights only.

def build_mobilenetv2_classifier(num_classes, input_shape=(224, 224, 3)):
    base = MobileNetV2(weights='imagenet', include_top=False, input_shape=input_shape)
    base.trainable = False

    inputs = layers.Input(shape=input_shape)
    x = tf.keras.applications.mobilenet_v2.preprocess_input(inputs)
    x = base(x, training=False)
    x = layers.GlobalAveragePooling2D()(x)
    x = layers.Dense(256, activation='relu')(x)
    x = layers.Dropout(0.5)(x)
    outputs = layers.Dense(num_classes, activation='softmax')(x)
    return models.Model(inputs, outputs, name='MobileNetV2')

try:
    model = tf.keras.models.load_model(MODEL_PATH, compile=False)
    print('Model loaded with tf.keras load_model')
except Exception as e:
    print('tf.keras load_model failed:', repr(e))
    print('Rebuilding MobileNetV2 classifier and loading weights...')
    model = build_mobilenetv2_classifier(num_classes)
    model.load_weights(MODEL_PATH)
    print('Weights loaded with model.load_weights')

print('Model ready:', model.name)

## 3) Pick one sample image (auto)

This cell grabs one image file from the dataset folder. If you want a specific image, set `IMG_PATH` manually.

In [ ]:
def find_first_image(dataset_dir):
    exts = ('.jpg', '.jpeg', '.png', '.webp')
    for root, dirs, files in os.walk(dataset_dir):
        for fn in files:
            if fn.lower().endswith(exts):
                return os.path.join(root, fn)
    return None

IMG_PATH = find_first_image(DATASET_DIR)
print('IMG_PATH:', IMG_PATH)

## 4) Predict

In [ ]:
img_bgr = cv2.imread(IMG_PATH)
img_rgb = cv2.cvtColor(img_bgr, cv2.COLOR_BGR2RGB)
img_resized = cv2.resize(img_rgb, IMG_SIZE)

x = np.expand_dims(img_resized.astype(np.float32), axis=0)
probs = model.predict(x, verbose=0)[0]
pred_idx = int(np.argmax(probs))
pred_name = class_names[pred_idx] if pred_idx < len(class_names) else str(pred_idx)
pred_conf = float(probs[pred_idx])

top3 = np.argsort(probs)[::-1][:3]
print('Prediction:', pred_name, f'({pred_conf:.3f})')
print('Top-3:')
for i in top3:
    print(' -', class_names[i], f'{float(probs[i]):.3f}')

plt.figure(figsize=(5,5))
plt.imshow(img_resized.astype('uint8'))
plt.title(f"Pred: {pred_name}\nConf: {pred_conf:.2%}")
plt.axis('off')
plt.show()

## 5) XAI Heatmap (Occlusion Sensitivity)

We slide a gray patch across the image. If occluding a region drops the predicted class score, that region is important.

You can control the resolution with `PATCH` and `STRIDE`:
- Smaller stride = smoother heatmap but slower.
- For a fast demo: PATCH=48, STRIDE=24.

In [ ]:
PATCH = 48
STRIDE = 24
FILL = 127.5

h, w = IMG_SIZE
base_prob = float(probs[pred_idx])

rows = (h - PATCH) // STRIDE + 1
cols = (w - PATCH) // STRIDE + 1
sens = np.zeros((rows, cols), dtype=np.float32)

for ri, y in enumerate(range(0, h - PATCH + 1, STRIDE)):
    for ci, x0 in enumerate(range(0, w - PATCH + 1, STRIDE)):
        occluded = img_resized.astype(np.float32).copy()
        occluded[y:y+PATCH, x0:x0+PATCH, :] = FILL
        p = model.predict(np.expand_dims(occluded, 0), verbose=0)[0]
        sens[ri, ci] = base_prob - float(p[pred_idx])

# Upsample to image size
sens_up = cv2.resize(sens, (w, h), interpolation=cv2.INTER_CUBIC)
sens_norm = (sens_up - sens_up.min()) / (sens_up.max() - sens_up.min() + 1e-8)

# Create overlay
heat = np.uint8(255 * sens_norm)
heat_color = cv2.applyColorMap(heat, cv2.COLORMAP_JET)
overlay_bgr = cv2.addWeighted(cv2.cvtColor(img_resized.astype('uint8'), cv2.COLOR_RGB2BGR), 0.60, heat_color, 0.40, 0)
overlay_rgb = cv2.cvtColor(overlay_bgr, cv2.COLOR_BGR2RGB)

plt.figure(figsize=(14, 4))
plt.subplot(1,3,1)
plt.imshow(img_resized.astype('uint8'))
plt.title('Original')
plt.axis('off')

plt.subplot(1,3,2)
plt.imshow(sens_norm, cmap='jet')
plt.title('XAI heatmap (occlusion)')
plt.axis('off')

plt.subplot(1,3,3)
plt.imshow(overlay_rgb)
plt.title('Overlay')
plt.axis('off')

plt.suptitle(f"Model: {os.path.basename(MODEL_PATH)} | Pred: {pred_name} ({pred_conf:.2%})", fontsize=11)
plt.tight_layout()

out_path = os.path.join(OUT_DIR, f"xai_occlusion_{os.path.basename(MODEL_PATH).replace('.h5','')}.png")
plt.savefig(out_path, dpi=200, bbox_inches='tight')
plt.show()

print('Saved:', out_path)